# 04 · Heart-rate track: tilt-table processing, label-gated classifier, label-free false-alarm test

**Track:** heart rate. This is the heart-rate half of the two-track design (oxygen is notebook 03).

**v3 (2026-11-09): no self-derived labels.** Earlier versions trained on "HR-collapse" events that this notebook defined from the HR signal itself. That label was invented here, never checked against the investigators' outcomes, and partly circular, because the benchmark rule *was* the label statistic. v3 removes it:
- Automatic HR-collapse detections are kept only as **candidates for clinician review** (a figure and a CSV). They are never used as training or evaluation labels.
- **Classifier training and cross-validation run only when labels with a stated origin are supplied** in `ADJUDICATED_EVENTS_CSV` (columns `subject,onset_s,source`). Valid sources are the investigators' own test outcomes or a clinician's adjudication. Without such labels, those cells print a skip message.
- The **label-free** analysis always runs. Nobody lost consciousness during swimming or the breath-hold study, so every alert there is a false alarm. Fact, not inference.

**Fainting data.** Head-up tilt tests, 128 recordings of patients with recurrent syncope ([dataset zip](https://www.sbng.si/file/2215652716491232_tilt-n-raziskava-anonimno2025-10-20.zip); [Klemenc et al., Frontiers in Neuroinformatics 2026](https://www.frontiersin.org/journals/neuroinformatics/articles/10.3389/fninf.2026.1740746/full)). ECG at 200 Hz plus a pressure channel, 55 min each. The modified Italian protocol is 5 min supine, 65° tilt, nitroglycerin after the 30-minute passive phase, and tilt-down about 15 minutes later.
- **The public release has no outcomes and no event times.** The article analysed 105 patients (52 HUTT-positive), but the per-patient outcome table is not published.
- Requesting it from the corresponding authors is the defensible route to labels. Their patient-level outcome plus a clinician-confirmed onset time gives exactly the CSV this notebook accepts.

**Swimming and breath-hold false-alarm data**

| Source | Content | Rate |
|---|---|---|
| [Wearable HR during swimming](https://zenodo.org/records/18992173) (CC BY 4.0) | 10 swimmers, Polar H10: dry rest, rest in water, swimming, running | 1 Hz |
| [Dry intermittent apnoea](https://zenodo.org/records/21701010) (CC BY 4.0) | 80 people: repeated breath-holds vs normal breathing; bradycardia without fainting | 1 Hz |

**Known gaps in the false-alarm data.**
- Every swimming trial runs rest → exercise and **stops when exercise stops**, so post-exercise HR recovery, when HR falls fastest, is absent.
- The breath-holds are dry. Face immersion strengthens the dive-reflex bradycardia, and no open dataset of it was found.

Zero alarms here therefore cannot rule out alarms in those situations.

**Claims this notebook cannot support:** detection of underwater hypoxic blackout. In drowning, loss of consciousness is caused by low oxygen, not by the vasovagal mechanism. The oxygen track (notebook 03) remains the primary physiological signal.

**Run on Google Colab:** keep the default CPU runtime and choose **Runtime → Run all**, authorising Google Drive when asked. Downloads (about 380 MB plus 2 MB) and processed recordings are cached in `MyDrive/KappaBetaML/cache/hr_syncope`. To skip downloading, upload the unchanged zip, `hr_dataset.csv` and `Supp database.xlsx` to `MyDrive/KappaBetaML/input/` and set `ALLOW_DOWNLOAD=False`. The tilt dataset page states no licence: keep copies private and cite the article. Outputs go to `MyDrive/KappaBetaML/outputs/hr_syncope`.

In [ ]:
import importlib.util
required = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'sklearn': 'scikit-learn', 'joblib': 'joblib',
            'openpyxl': 'openpyxl', 'requests': 'requests', 'matplotlib': 'matplotlib', 'IPython': 'ipython'}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ImportError('Missing packages: ' + ', '.join(missing) + '. Install only these with %pip, restart and Run All.')
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import hashlib, json, math, os, platform, re, time, zipfile
import numpy as np
import pandas as pd
import openpyxl, requests, joblib, sklearn
from scipy import signal, stats
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt
from IPython.display import display

def env_flag(name, default):
    return os.getenv(name, str(default)).lower() in {'1', 'true', 'yes'}

# ---- Storage: Google Colab ----
# Colab's /content disk is wiped when the runtime ends, so outputs (and the downloads cache for small
# sources) go to Google Drive by default. Set SAVE_TO_DRIVE = False to keep everything in this runtime.
import sys
ON_COLAB = 'google.colab' in sys.modules
SAVE_TO_DRIVE = env_flag('KB_SAVE_TO_DRIVE', ON_COLAB)
if ON_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/KappaBetaML')
else:
    BASE = Path('/content') if ON_COLAB else Path('.')
DATA_ROOT = Path(os.getenv('HRSYNC_DATA_ROOT', str(BASE / 'input')))   # optional manually uploaded originals
OUT = Path(os.getenv('HRSYNC_OUTPUT_DIR', str(BASE / 'outputs' / 'hr_syncope')))
FIGURES = OUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
CACHE = Path(os.getenv('HRSYNC_CACHE', str(BASE / 'cache' / 'hr_syncope')))
ALLOW_DOWNLOAD = env_flag('HRSYNC_DOWNLOAD', True)
# Labels with a stated origin only: subject,onset_s,source (blank onset = test negative / no syncope).
ADJUDICATED_EVENTS_CSV = os.getenv('HRSYNC_ADJUDICATED_CSV', '')
WORKERS = int(os.getenv('HRSYNC_WORKERS', str(min(4, os.cpu_count() or 1))))   # free Colab: 2 vCPUs
SEED = 20261108

# ---- Protocol and evaluation settings, fixed for v3. Not pre-registered: v1/v2 results on these data were seen. ----
FS = 200                         # tilt recordings, Hz (stated by the dataset article)
MIN_RECORDING_S = 2700           # exclude truncated recordings (< 45 min)
SUPINE = (60, 290)               # baseline seconds; tilt assumed at 300 s (protocol)
SEARCH = (480, 2940)             # events searched from 3 min after tilt to before the ~50 min tilt-down
HR_COLLAPSE_FALL = 0.70          # REVIEW CANDIDATES ONLY (never labels): 10-s median HR <= 70% of 180-s peak ...
HR_COLLAPSE_HOLD_S = 10          # ... and <= supine baseline, sustained for 10 s
BP_CONFIRM_RATIO = 0.80          # descriptor only: artifact-filtered SBP <= 80% of its local reference near onset
HORIZON_S = 60                   # positive anchors: onset within the next 60 s (or ongoing up to LATE_GRACE_S)
LATE_GRACE_S = 15                # alerts up to 15 s after onset count as late detections
CREDIT_LEAD_S = 180              # alerts opening more than 180 s before onset are unmatched
MAX_UNMATCHED_PER_HOUR = 2.0     # validation alert budget
MAX_WARNING_FRACTION = 0.10
TRAIN_STRIDE_S = 5
N_FOLDS = 5
MIN_LABELLED_EVENTS = 10         # fewer labelled events (or non-events) than this: no training / cross-validation
# Published reference point for the label-free test: VASIS type 2A (cardioinhibitory) is HR < 40 bpm for > 10 s.
VASIS_HR_BPM, VASIS_HOLD_S = 40.0, 10
# Label-free threshold sweeps (alert rate as a function of threshold; no threshold is chosen here).
SWEEP_GRID = {'rule_drop_from_peak': [0.15, 0.20, 0.25, 0.30, 0.35, 0.40],
              'rule_drop_from_baseline': [0.10, 0.15, 0.20, 0.25, 0.30, 0.40],
              'cusum': [1.0, 2.0, 3.0, 4.0, 6.0]}

def poisson_rate_ci(count, hours, level=0.95):
    '''Exact (Garwood) CI for alerts per hour. Treats alert starts as independent, so it understates the
    width when a few people produce most alerts.'''
    a = 1 - level
    lo = 0.0 if count == 0 else stats.chi2.ppf(a / 2, 2 * count) / 2
    return lo / hours, stats.chi2.ppf(1 - a / 2, 2 * count + 2) / 2 / hours

def proportion_ci(k, n, level=0.95):
    '''Wilson CI for a proportion such as event recall.'''
    if n == 0:
        return np.nan, np.nan
    ci = stats.binomtest(int(k), int(n)).proportion_ci(confidence_level=level, method='wilson')
    return ci.low, ci.high
print('Output:', OUT.resolve(), '| workers:', WORKERS)

## Acquire and verify sources

All three sources are pinned by MD5. The tilt zip's MD5 equals the server ETag observed on 2026-11-08. A mismatch stops the run, because a changed source needs review.

In [ ]:
SOURCES = {
    'tilt': ('tilt-n-raziskava-anonimno2025-10-20.zip',
             'https://www.sbng.si/file/2215652716491232_tilt-n-raziskava-anonimno2025-10-20.zip',
             'c62d7a690aec4885c2b9d7517cbd5770'),
    'swim': ('hr_dataset.csv', 'https://zenodo.org/records/18992173/files/hr_dataset.csv?download=1',
             '5ef843ff2fe44a164d6f1d01d9f8db6a'),
    'apnea': ('Supp database.xlsx', 'https://zenodo.org/records/21701010/files/Supp%20database.xlsx?download=1',
              '79fc9abbd6e3e04c52d17df3247751ec'),
}

def md5(path, chunk=1 << 22):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def acquire(name, url, digest):
    found = list(DATA_ROOT.rglob(name)) if DATA_ROOT.is_dir() else []
    # An uploaded zip may have been renamed; accept any large zip with the pinned digest.
    if name.endswith('.zip') and DATA_ROOT.is_dir():
        found += [p for p in DATA_ROOT.rglob('*.zip') if p.stat().st_size > 300e6]
    for path in [*found, CACHE / name]:
        if path.is_file() and md5(path) == digest:
            return path
    if not ALLOW_DOWNLOAD:
        raise FileNotFoundError(f'Attach the original {name} below {DATA_ROOT} or enable Internet.')
    CACHE.mkdir(parents=True, exist_ok=True)
    part = CACHE / (name + '.part')
    for attempt in range(4):
        try:
            offset = part.stat().st_size if part.exists() else 0
            headers = {'User-Agent': 'hr-syncope-notebook/1.0', **({'Range': f'bytes={offset}-'} if offset else {})}
            with requests.get(url, stream=True, timeout=(15, 120), headers=headers) as r:
                r.raise_for_status()
                mode = 'ab' if offset and r.status_code == 206 else 'wb'
                with open(part, mode) as f:
                    for block in r.iter_content(1 << 20):
                        f.write(block)
            if md5(part) != digest:
                part.unlink()
                raise ValueError('Checksum mismatch; the source changed. Review before use.')
            part.replace(CACHE / name)
            return CACHE / name
        except (requests.RequestException, OSError, ValueError) as error:
            last = error
            time.sleep(2 + 2 * attempt)
    raise RuntimeError(f'Cannot acquire {name}: {last}')

paths = {key: acquire(*spec) for key, spec in SOURCES.items()}
print({k: f'{v.name} ({v.stat().st_size / 1e6:.1f} MB)' for k, v in paths.items()})

## Signal processing: ECG → heart rate, pressure → systolic envelope

Each recording is a tab-separated text file with a pressure column and two ECG columns. The header gives a per-session pressure calibration `P = a·p + b`. In most files `a = 1`, meaning the values are in **uncalibrated device units**, so pressure is only used *relative to the same recording*.

The processing steps are:
- **ECG:** band-pass at 5–25 Hz, take derivative energy, detect peaks with an adaptive threshold every 30 s, then refine each R-peak.
- **Beat cleaning:** discard RR intervals outside 0.3–2.0 s, and any that differ by more than 25% from the local median of 9 beats.
- **Lead choice:** use the lead giving more valid one-second HR values. This uses no labels.
- **1 Hz HR:** the mean instantaneous HR of the beats ending in each second. This matches the wearable's one-value-per-second contract.
- **Pressure:** take the per-second maximum and minimum as the systolic/diastolic envelope.

Processed arrays are cached, so re-runs skip the parsing.

In [ ]:
def read_recording(zip_path, member):
    with zipfile.ZipFile(zip_path) as z, z.open(member) as f:
        head = [f.readline().decode('latin1') for _ in range(4)]
        frame = pd.read_csv(f, sep='\t', header=None, names=['p', 'e1', 'e2'], dtype=np.float32, engine='c')
    m = re.search(r'P = ([\d.]+)\*p ([+-]) ([\d.]+)', head[1])
    calibration = (float(m.group(1)), float(m.group(3)) * (1 if m.group(2) == '+' else -1)) if m else (np.nan, np.nan)
    return frame, calibration

def r_peaks(ecg):
    sos = signal.butter(3, [5, 25], fs=FS, btype='band', output='sos')
    x = signal.sosfiltfilt(sos, np.asarray(ecg, float))
    width = int(0.08 * FS)
    energy = np.convolve(np.diff(x, prepend=x[0]) ** 2, np.ones(width) / width, mode='same')
    peaks = []
    for start in range(0, len(energy), 30 * FS):
        block = energy[start:start + 30 * FS]
        if len(block) < FS:
            continue
        found, _ = signal.find_peaks(block, height=0.25 * np.percentile(block, 99), distance=int(0.3 * FS))
        peaks.extend(start + found)
    radius = int(0.06 * FS)
    refined = [max(0, p - radius) + int(np.argmax(np.abs(x[max(0, p - radius):p + radius + 1]))) for p in peaks]
    return np.unique(np.asarray(refined, int))

def clean_beats(peaks):
    t = peaks / FS
    rr = np.diff(t)
    beat_t = t[1:]
    median = pd.Series(rr).rolling(9, center=True, min_periods=3).median().to_numpy()
    ok = (rr > 0.3) & (rr < 2.0) & (np.abs(rr - median) < 0.25 * median)
    return beat_t[ok], rr[ok]

def hr_per_second(beat_t, rr, n_seconds):
    hr = np.full(n_seconds, np.nan)
    second = np.floor(beat_t).astype(int)
    keep = (second >= 0) & (second < n_seconds)
    means = pd.Series(60.0 / rr[keep]).groupby(second[keep]).mean()
    hr[means.index.to_numpy()] = means.to_numpy()
    return hr

def process_member(args):
    zip_path, member, cache_dir = args
    target = Path(cache_dir) / (member.rsplit('.', 1)[0] + '.npz')
    if target.exists():
        return member, str(target)
    frame, (a, b) = read_recording(zip_path, member)
    n = len(frame) // FS
    best = None
    for lead in ['e1', 'e2']:
        beat_t, rr = clean_beats(r_peaks(frame[lead].to_numpy()))
        hr = hr_per_second(beat_t, rr, n)
        coverage = np.isfinite(hr).mean()
        if best is None or coverage > best[0]:
            best = (coverage, lead, hr, beat_t, rr)
    _, lead, hr, beat_t, rr = best
    pressure = frame.p.to_numpy()[:n * FS].reshape(n, FS)
    np.savez_compressed(target, hr=hr, sbp=pressure.max(1), dbp=pressure.min(1), beat_t=beat_t, rr=rr,
                        cal_a=a, cal_b=b, lead=lead)
    return member, str(target)

processed_dir = CACHE / 'tilt_processed'
processed_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(paths['tilt']) as z:
    members = sorted(m for m in z.namelist() if m.lower().endswith('.txt'))
jobs = [(str(paths['tilt']), m, str(processed_dir)) for m in members]
started = time.monotonic()
if WORKERS > 1 and platform.system() == 'Linux':     # fork start method: notebook functions are picklable
    with ProcessPoolExecutor(WORKERS) as pool:
        done = list(pool.map(process_member, jobs))
else:
    done = [process_member(job) for job in jobs]
print(f'Processed {len(done)} recordings in {time.monotonic() - started:.0f} s')

recordings, audit_rows = {}, []
for member, npz in done:
    d = np.load(npz)
    subject = 'tilt/' + member.split('-')[0]          # birth-year suffix dropped from outputs
    n = len(d['hr'])
    tilt_cov = float(np.isfinite(d['hr'][SUPINE[0]:min(n, SEARCH[1])]).mean()) if n > SUPINE[0] else 0.0
    reason = ('short' if n < MIN_RECORDING_S else 'low_hr_coverage' if tilt_cov < 0.8 else '')
    audit_rows.append({'subject': subject, 'seconds': n, 'hr_coverage': round(tilt_cov, 3),
                       'pressure_calibrated': bool(float(d['cal_a']) != 1.0), 'lead': str(d['lead']), 'excluded': reason})
    if not reason:
        recordings[subject] = {k: d[k] for k in ['hr', 'sbp', 'dbp', 'beat_t', 'rr']}
recording_audit = pd.DataFrame(audit_rows)
print(recording_audit.excluded.replace('', 'included').value_counts().to_string())

## Labels: only from the investigators or a clinician; automatic detections are review candidates

**Accepted labels.** `ADJUDICATED_EVENTS_CSV` has one row per recording, with columns `subject`, `onset_s` and `source`:
- `subject`: for example `001` or `tilt/001`.
- `onset_s`: seconds from recording start. A blank value means the test was negative, with no syncope.
- `source`: who determined the label, such as "Klemenc et al. outcome table, onset confirmed by Dr X" or "clinician adjudication from ECG/BP review, 2026-11-20". Rows without a source are rejected.

Recordings with **no row are excluded** from the labelled analysis. *Unknown* is not the same as *negative*.

**Review candidates.** `hr_collapse_onset` flags the first sustained fall of the 10-s median HR to ≤ 70% of its 180-s peak and below the supine baseline. These flags exist only to speed up clinician review: see the candidate-review figure and `review_candidates.csv`.
- **They are not labels.** The criterion was chosen by us, not taken from a clinical definition.
- **They miss vasodepressor faints.** The heart rate stays up in those.
- **Some may be tilt-down.** The tilt-down time is assumed from the protocol, not annotated.

**SBP-fall descriptor.** For labelled events, the notebook records whether artifact-filtered systolic pressure fell to ≤ 80% of its local reference near onset. This lets results be split by haemodynamic pattern; it is not a label.

In [ ]:
def hr_collapse_onset(hr):
    '''Review candidate only (never a label): first sustained precipitous HR fall in the search window.'''
    smooth = pd.Series(hr, dtype=float).rolling(10, min_periods=6).median()
    peak = smooth.shift(1).rolling(180, min_periods=90).max()
    baseline = np.nanmedian(smooth.to_numpy()[SUPINE[0]:SUPINE[1]])
    condition = ((smooth <= HR_COLLAPSE_FALL * peak) & (smooth <= baseline)).to_numpy()
    run = 0
    for i in range(SEARCH[0], min(SEARCH[1], len(condition))):
        run = run + 1 if condition[i] else 0
        if run >= HR_COLLAPSE_HOLD_S:
            return i - HR_COLLAPSE_HOLD_S + 1
    return None

def sbp_ratio(sbp, dbp):
    s, d = pd.Series(sbp, dtype=float), pd.Series(dbp, dtype=float)
    pp = s - d
    base_pp, base_s = np.nanmedian(pp[SUPINE[0]:SUPINE[1]]), np.nanmedian(s[SUPINE[0]:SUPINE[1]])
    valid = (pp > 0.3 * base_pp) & (pp < 2.5 * base_pp) & (s > 0.3 * base_s) & (s < 1.8 * base_s)
    centred = s.where(valid).rolling(31, center=True, min_periods=10).median()   # offline descriptor only
    valid &= (s - centred).abs() <= 0.25 * centred
    smooth = s.where(valid).rolling(15, min_periods=10).median()
    reference = smooth.shift(60).rolling(180, min_periods=90).median()
    return (smooth / reference).to_numpy()

def normalise_subject(value):
    value = str(value).strip()
    return value if value.startswith('tilt/') else 'tilt/' + value.split('-')[0].zfill(3)

investigator_labels = {}
if ADJUDICATED_EVENTS_CSV:
    table = pd.read_csv(ADJUDICATED_EVENTS_CSV, dtype={'subject': str, 'source': str})
    if not {'subject', 'onset_s', 'source'} <= set(table):
        raise ValueError('Label CSV needs columns subject,onset_s,source (source = who determined the label).')
    if table.source.fillna('').str.strip().eq('').any():
        raise ValueError('Every label row needs a source; labels without a stated origin are not accepted.')
    investigator_labels = {normalise_subject(r.subject): (None if pd.isna(r.onset_s) else int(r.onset_s), r.source)
                           for r in table.itertuples()}
    absent = sorted(set(investigator_labels) - set(recordings))
    if absent:
        print('Label rows for recordings excluded or absent here:', absent)

def sbp_fall_near(rec, at):
    if at is None:
        return False
    window = rec['sbp_ratio'][max(0, at - 120):at + 60]
    return bool(np.nanmin(window) <= BP_CONFIRM_RATIO) if np.isfinite(window).any() else False

candidate_rows = []
for subject, rec in recordings.items():
    rec['baseline'] = float(np.nanmedian(rec['hr'][SUPINE[0]:SUPINE[1]]))
    rec['sbp_ratio'] = sbp_ratio(rec['sbp'], rec['dbp'])
    candidate = hr_collapse_onset(rec['hr'])
    labelled = subject in investigator_labels
    onset, source = investigator_labels.get(subject, (None, ''))
    rec.update(candidate=candidate, labelled=labelled, onset=onset, bp_confirmed=sbp_fall_near(rec, onset))
    candidate_rows.append({'subject': subject, 'review_candidate_onset_s': candidate,
                           'review_candidate_min': None if candidate is None else round(candidate / 60, 2),
                           'sbp_fall_near_candidate': sbp_fall_near(rec, candidate), 'labelled': labelled,
                           'label_onset_s': onset, 'label_source': source, 'baseline_hr': round(rec['baseline'], 1)})
candidates = pd.DataFrame(candidate_rows)
labels = (candidates[candidates.labelled]
          .assign(event=lambda d: d.label_onset_s.notna(), onset_s=lambda d: d.label_onset_s,
                  bp_confirmed=lambda d: [recordings[s]['bp_confirmed'] for s in d.subject])
          [['subject', 'event', 'onset_s', 'bp_confirmed', 'label_source']].reset_index(drop=True))
LABELS_AVAILABLE = bool(len(labels)) and labels.event.sum() >= MIN_LABELLED_EVENTS \
    and (~labels.event).sum() >= MIN_LABELLED_EVENTS
print(f"Review candidates (not labels): {candidates.review_candidate_onset_s.notna().sum()} of {len(candidates)} recordings.")
print(f"Labelled recordings: {len(labels)} ({int(labels.event.sum()) if len(labels) else 0} with syncope); "
      f"classifier training {'ENABLED' if LABELS_AVAILABLE else 'SKIPPED'} "
      f"(needs >= {MIN_LABELLED_EVENTS} labelled events and non-events with a stated source).")

## Features and architectures

All features are causal at second *t* and use only past data. HR is normalised by the person's **resting baseline**. In the tilt test that is the supine period; in water it would be a pre-swim rest.

| Feature set | Inputs | Device requirement |
|---|---|---|
| `hr1hz` | Last 10-s median HR; ratio to baseline; ratio to 180-s peak; peak/baseline (prior tachycardia); 10-s and 30-s slopes; 30-s and 120-s SD; 120-s range; 120-s mean/baseline | Current 1 Hz HR |
| `hr1hz+hrv` | Above, plus RMSSD and SDNN over 60 s (relative to mean RR) and beat count | Beat timing (100 Hz PPG) |
| `window` | The last 120 s of HR/baseline, in 2-s bins | Current 1 Hz HR |

| Architecture | Type |
|---|---|
| `no_alert` | Constant baseline |
| `rule_absolute_hr` | Fixed-type rule: lower HR, more concern |
| `rule_drop_from_peak` | HR relative to its recent peak. This mirrors the label definition: the benchmark the learned models must beat. |
| `rule_drop_from_baseline` | HR relative to the resting baseline |
| `cusum` | One-sided CUSUM of relative HR falls below the trailing median |
| `logistic` / `boosted_trees` | Learned on `hr1hz` |
| `logistic_hrv` / `boosted_trees_hrv` | Learned on `hr1hz+hrv` |
| `mlp_window` | Small neural network on the raw 120-s window |
| `isolation_forest` | Normal-only anomaly model, trained on tilt windows far from any event |

Every architecture outputs a score where higher means riskier. All of them use **the same threshold-selection procedure and the same alert accounting**.

In [ ]:
HR1HZ = ['hr_med10', 'rel_baseline', 'rel_peak180', 'peak_over_baseline', 'slope10', 'slope30',
         'sd30', 'sd120', 'range120_rel', 'mean120_rel']
HRV = ['rmssd60_rel', 'sdnn60_rel', 'beats60']
WINDOW_BINS = 60

def hr_features(hr, baseline, beat_t=None, rr=None):
    '''Causal per-second features from a 1 Hz HR array (NaN = missing).'''
    h = pd.Series(np.asarray(hr, float))
    med10 = h.rolling(10, min_periods=6).median()
    peak180 = med10.shift(1).rolling(180, min_periods=90).max()
    filled = med10.ffill(limit=5)
    f = pd.DataFrame({
        'hr_med10': med10, 'rel_baseline': med10 / baseline, 'rel_peak180': med10 / peak180,
        'peak_over_baseline': peak180 / baseline,
        'slope10': (filled - filled.shift(10)) / 10 / baseline, 'slope30': (filled - filled.shift(30)) / 30 / baseline,
        'sd30': h.rolling(30, min_periods=20).std() / baseline, 'sd120': h.rolling(120, min_periods=80).std() / baseline,
        'range120_rel': (med10.rolling(120, min_periods=80).max() - med10.rolling(120, min_periods=80).min()) / baseline,
        'mean120_rel': h.rolling(120, min_periods=80).mean() / baseline,
        'valid30': h.notna().astype(float).rolling(30, min_periods=1).mean(),
    })
    if beat_t is not None and len(beat_t) > 3:
        seconds = np.arange(len(h)) + 1.0
        hi = np.searchsorted(beat_t, seconds, side='right')
        lo = np.searchsorted(beat_t, seconds - 60, side='right')
        c1, c2 = np.r_[0, np.cumsum(rr)], np.r_[0, np.cumsum(rr ** 2)]
        d2 = np.r_[0, np.cumsum(np.diff(rr) ** 2)]
        n = hi - lo
        with np.errstate(invalid='ignore', divide='ignore'):
            mean = (c1[hi] - c1[lo]) / n
            sdnn = np.sqrt(np.maximum((c2[hi] - c2[lo]) / n - mean ** 2, 0))
            pairs = np.maximum(n - 1, 0)
            rmssd = np.sqrt((d2[np.maximum(hi - 1, 0)] - d2[np.minimum(lo, np.maximum(hi - 1, 0))]) / pairs)
        ok = n >= 20
        f['rmssd60_rel'] = np.where(ok, rmssd / mean, np.nan)
        f['sdnn60_rel'] = np.where(ok, sdnn / mean, np.nan)
        f['beats60'] = np.where(ok, n, np.nan)
    else:
        for c in HRV:
            f[c] = np.nan
    return f

def window_matrix(hr, baseline, seconds):
    '''Last 120 s of HR/baseline in 2-s bins for the requested anchor seconds.'''
    rel = pd.Series(np.asarray(hr, float) / baseline).ffill(limit=10).fillna(1.0).to_numpy(np.float32)
    padded = np.r_[np.ones(120, np.float32), rel]
    windows = sliding_window_view(padded, 120)[np.asarray(seconds, int) + 1]   # rel[s-119 .. s]
    return windows.reshape(len(windows), WINDOW_BINS, 2).mean(2)

def cusum_score(hr):
    med10 = pd.Series(np.asarray(hr, float)).rolling(10, min_periods=6).median()
    reference = med10.shift(1).rolling(180, min_periods=90).median().to_numpy()
    x = med10.to_numpy()
    s, out = 0.0, np.full(len(x), np.nan)
    for i in range(len(x)):
        if np.isfinite(x[i]) and np.isfinite(reference[i]):
            s = max(0.0, s + (1 - x[i] / reference[i]) - 0.05)
            out[i] = s
    return out

def anchors_for(subject, rec):
    f = hr_features(rec['hr'], rec['baseline'], rec['beat_t'], rec['rr'])
    f['cusum'] = cusum_score(rec['hr'])
    onset = rec['onset']
    end = SEARCH[1] if onset is None else min(SEARCH[1], onset + LATE_GRACE_S + 1)
    t = np.arange(SEARCH[0], min(end, len(f)))
    f = f.iloc[t].copy()
    f['t'] = t
    f['subject'] = subject
    f['target'] = 0 if onset is None else ((t >= onset - HORIZON_S) & (t <= onset + LATE_GRACE_S)).astype(int)
    f['onset'] = -1 if onset is None else onset
    f['bp_confirmed'] = rec['bp_confirmed']
    eligible = f[HR1HZ].notna().all(axis=1) & (f.valid30 >= 0.7)
    return f[eligible].reset_index(drop=True)

if LABELS_AVAILABLE:
    anchors = pd.concat([anchors_for(s, r) for s, r in recordings.items() if r['labelled']], ignore_index=True)
    print(f"{len(anchors):,} eligible anchor seconds from {anchors.subject.nunique()} labelled recordings; "
          f"positive anchors {anchors.target.sum():,}; HRV available {anchors[HRV].notna().all(axis=1).mean():.0%}")
else:
    print('Feature functions defined; no labelled anchors built (no investigator/clinician labels).')

*Runs only with investigator/clinician labels.*

## Evaluation protocol: patient-grouped cross-validation with event-level alerts

The protocol has five patient-grouped folds, stratified by event presence:
- **Fitting:** within each training fold, 75% of patients fit the models.
- **Thresholds:** the other 25% choose each threshold. The rule maximises the number of credited events, subject to ≤ 2 unmatched alert starts per hour and ≤ 10% of time in warning. Ties go to fewer unmatched starts, then the higher threshold. If nothing qualifies, alerts are disabled.
- **Testing:** the held-out fold is scored once.

**Alert accounting**
- An alert opens when the score crosses its threshold and stays open while the score remains above it.
- An event is credited if an alert opens between 180 s before and 15 s after onset. Each event is credited once.
- **Lead time** is onset minus alert start; a negative lead means the alert was late.
- Every other alert start is unmatched.
- Overlapping seconds are not independent, so the tables report event and patient counts.

In [ ]:
RULES = {
    'no_alert': lambda f: np.zeros(len(f)),
    'rule_absolute_hr': lambda f: -f.hr_med10.to_numpy(),
    'rule_drop_from_peak': lambda f: 1 - f.rel_peak180.to_numpy(),
    'rule_drop_from_baseline': lambda f: 1 - f.rel_baseline.to_numpy(),
    'cusum': lambda f: f.cusum.fillna(0).to_numpy(),
}
LEARNED = {
    'logistic': HR1HZ, 'boosted_trees': HR1HZ, 'logistic_hrv': HR1HZ + HRV, 'boosted_trees_hrv': HR1HZ + HRV,
    'mlp_window': 'window', 'isolation_forest': HR1HZ,
}

def patient_weights(frame):
    counts = frame.groupby('subject').subject.transform('size')
    w = 1.0 / counts
    return (w / w.mean()).to_numpy()

def design(frame, cols, source=None):
    if cols != 'window':
        return frame[cols].to_numpy(float)
    source = recordings if source is None else source
    X = np.empty((len(frame), WINDOW_BINS), np.float32)
    subject = frame.subject.to_numpy()
    t = frame.t.to_numpy()
    for s in pd.unique(subject):
        rows = np.flatnonzero(subject == s)
        X[rows] = window_matrix(source[s]['hr'], source[s]['baseline'], t[rows])
    return X

def fit_model(name, train):
    cols = LEARNED[name]
    data = train[train.t.mod(TRAIN_STRIDE_S).eq(0)]
    if cols != 'window' and name.endswith('_hrv'):
        data = data[data[HRV].notna().all(axis=1)]
    if name == 'isolation_forest':
        far = data[(data.onset < 0) | (data.t < data.onset - 600)]
        return IsolationForest(n_estimators=200, random_state=SEED).fit(design(far, cols))
    X, y, w = design(data, cols), data.target.to_numpy(), patient_weights(data)
    if name.startswith('logistic'):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
        return model.fit(X, y, logisticregression__sample_weight=w)
    if name.startswith('boosted'):
        return HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=15, learning_rate=0.05,
                                              min_samples_leaf=40, l2_regularization=1.0, random_state=SEED).fit(X, y, sample_weight=w)
    # MLPClassifier has no sample weights here: rebalance by repeating positive anchors (about 1:4),
    # and standardise inputs. Accuracy-based early stopping would stop at the all-negative solution.
    positives, negatives = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    repeat = max(1, round(len(negatives) / (4 * max(1, len(positives)))))
    rows = np.r_[negatives, np.repeat(positives, repeat)]
    model = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3, max_iter=60,
                                                          random_state=SEED))
    return model.fit(X[rows], y[rows])

def score(name, model, frame, source=None):
    if name in RULES:
        return RULES[name](frame)
    cols = LEARNED[name]
    out = np.full(len(frame), np.nan)
    usable = np.ones(len(frame), bool) if cols == 'window' else frame[cols].notna().all(axis=1).to_numpy()
    if usable.any():
        X = design(frame[usable], cols, source)
        out[usable] = -model.score_samples(X) if name == 'isolation_forest' else model.predict_proba(X)[:, 1]
    return out

def prepare(frame):
    '''Arrays for fast event-level accounting; frame rows must be in (subject, t) order.'''
    subject = frame.subject.to_numpy()
    t = frame.t.to_numpy()
    onset = frame.onset.to_numpy()
    return {'subject': subject, 't': t, 'onset': onset, 'rows': len(frame),
            'same': np.r_[False, (subject[1:] == subject[:-1]) & (np.diff(t) == 1)],
            'in_credit': (onset >= 0) & (t >= onset - CREDIT_LEAD_S) & (t <= onset + LATE_GRACE_S)}

def alert_outcome(frame, values, threshold):
    '''Event-level accounting: an alert opens on a crossing and stays open while above threshold.'''
    p = frame if isinstance(frame, dict) else prepare(frame)
    high = np.zeros(p['rows'], bool) if threshold is None else np.nan_to_num(values, nan=-np.inf) >= threshold
    start = high & ~(np.r_[False, high[:-1]] & p['same'])
    idx = np.flatnonzero(start & p['in_credit'])
    first = pd.Series(p['t'][idx]).groupby(p['subject'][idx]).min() if len(idx) else pd.Series(dtype=float)
    onset_of = dict(zip(p['subject'][idx], p['onset'][idx]))
    leads = [int(onset_of[s] - t) for s, t in first.items()]
    starts = int(start.sum())
    hours = p['rows'] / 3600
    return {'alert_starts': starts, 'unmatched': starts - len(first),
            'unmatched_per_hour': (starts - len(first)) / hours if hours else np.nan,
            'warning_fraction': float(high.mean()) if p['rows'] else np.nan,
            'credited': set(first.index), 'leads': leads}

def choose_threshold(frame, values):
    events = frame.loc[frame.onset >= 0, 'subject'].nunique()
    if events == 0 or not np.isfinite(values).any():
        return None
    p = prepare(frame)
    finite = values[np.isfinite(values)]
    candidates = np.unique(np.quantile(finite, np.r_[np.linspace(0.5, 0.99, 99), np.linspace(0.99, 1, 51)]))
    best, threshold = (0, 0, -np.inf), None
    for c in candidates:
        o = alert_outcome(p, values, c)
        if o['unmatched_per_hour'] > MAX_UNMATCHED_PER_HOUR or o['warning_fraction'] > MAX_WARNING_FRACTION:
            continue
        rank = (len(o['credited']), -o['unmatched'], c)
        if rank > best:
            best, threshold = rank, float(c)
    return threshold if best[0] > 0 else None

In [ ]:
if not LABELS_AVAILABLE:
    print('Skipped: no investigator/clinician labels supplied (see the Labels section).')
else:
    subjects = labels.set_index('subject')
    rng = np.random.default_rng(SEED)
    fold_of = {}
    for has_event, group in subjects.groupby('event'):
        ids = rng.permutation(group.index.to_numpy())
        fold_of.update({s: i % N_FOLDS for i, s in enumerate(ids)})
    anchors['fold'] = anchors.subject.map(fold_of)
    MODELS = list(RULES) + list(LEARNED)
    oof = {m: np.full(len(anchors), np.nan) for m in MODELS}
    fold_thresholds, fold_rows = [], []
    started = time.monotonic()
    for fold in range(N_FOLDS):
        test_mask = anchors.fold.eq(fold).to_numpy()
        train_ids = np.array(sorted(set(anchors.subject[~test_mask])))
        tune_ids = set()
        for has_event in [True, False]:
            pool = [s for s in train_ids if bool(subjects.loc[s, 'event']) == has_event]
            tune_ids |= set(rng.permutation(pool)[:max(1, round(0.25 * len(pool)))])
        fit_part = anchors[~test_mask & ~anchors.subject.isin(tune_ids)]
        tune_part = anchors[anchors.subject.isin(tune_ids)].reset_index(drop=True)
        test_part = anchors[test_mask]
        for m in MODELS:
            model = None if m in RULES else fit_model(m, fit_part)
            th = None if m == 'no_alert' else choose_threshold(tune_part, score(m, model, tune_part))
            oof[m][test_mask] = score(m, model, test_part)
            fold_thresholds.append({'fold': fold, 'model': m, 'threshold': th})
        print(f'fold {fold + 1}/{N_FOLDS} done ({time.monotonic() - started:.0f} s)', flush=True)
    thresholds = pd.DataFrame(fold_thresholds)

In [ ]:
if not LABELS_AVAILABLE:
    print('Skipped: no investigator/clinician labels supplied (see the Labels section).')
else:
    def evaluate(frame, values_by_model, threshold_lookup, subset_name):
        rows = []
        for m in MODELS:
            values = values_by_model[m]
            per_fold_high = np.zeros(len(frame), bool)
            starts = unmatched = 0
            credited, leads = set(), []
            for fold in range(N_FOLDS):
                take = frame.fold.eq(fold).to_numpy()
                th = threshold_lookup.get((fold, m))
                o = alert_outcome(frame[take].reset_index(drop=True), values[take], th)
                starts += o['alert_starts']; unmatched += o['unmatched']; credited |= o['credited']; leads += o['leads']
                per_fold_high[take] = False if th is None else np.nan_to_num(values[take], nan=-np.inf) >= th
            y = frame.target.to_numpy()
            finite = np.isfinite(values)
            events = frame.loc[frame.onset >= 0, 'subject'].nunique()
            both = finite.any() and len(np.unique(y[finite])) == 2
            early = sum(l > 0 for l in leads)
            recall_lo, recall_hi = proportion_ci(len(credited), events)
            rate_lo, rate_hi = poisson_rate_ci(unmatched, len(frame) / 3600)
            rows.append({'subset': subset_name, 'model': m, 'patients': frame.subject.nunique(), 'events': events,
                         'detected': len(credited), 'event_recall': len(credited) / events if events else np.nan,
                         'recall_ci95_lo': recall_lo, 'recall_ci95_hi': recall_hi,
                         'unmatched_ci95_lo': rate_lo, 'unmatched_ci95_hi': rate_hi,
                         'early_warnings': early, 'median_lead_s': float(np.median(leads)) if leads else np.nan,
                         'unmatched_per_hour': unmatched / (len(frame) / 3600), 'warning_fraction': float(per_fold_high.mean()),
                         'window_AUROC': roc_auc_score(y[finite], values[finite]) if both else np.nan,
                         'window_AP': average_precision_score(y[finite], values[finite]) if both else np.nan,
                         'folds_with_alerts': int(thresholds[(thresholds.model == m)].threshold.notna().sum())})
        return pd.DataFrame(rows)

    lookup = {(r.fold, r.model): (None if pd.isna(r.threshold) else r.threshold) for r in thresholds.itertuples()}
    results = evaluate(anchors, oof, lookup, 'all events')
    confirmed_frame = anchors[(anchors.onset < 0) | anchors.bp_confirmed].reset_index(drop=True)
    confirmed_values = {m: oof[m][((anchors.onset < 0) | anchors.bp_confirmed).to_numpy()] for m in MODELS}
    results = pd.concat([results, evaluate(confirmed_frame, confirmed_values, lookup, 'events with SBP fall')], ignore_index=True)
    display(results.round(3))

## Label-free test: false alarms during swimming and breath-holds

This section needs no labels. In both datasets **nobody lost consciousness**, so every alert start is a false alarm.
- **Swimming dataset:** Polar H10 1 Hz HR. Each swimmer's baseline is the median of their *dry rest* trial. Trials are split at timestamp gaps.
- **Breath-hold dataset:** Polar H10 1 Hz HR. The baseline is the study's recorded resting HR. Sessions are 6 minutes long, so only the last 3 minutes have the full 180-s history.

**No threshold is chosen here.** Each rule is swept over a grid, and the table reports the false-alarm rate at every threshold, with exact Poisson 95% intervals. One published point is included: **HR < 40 bpm for > 10 s**, the VASIS type 2A (cardioinhibitory) criterion.

Exposure is short: about 1 hour of swimming and 0.3 hours of rest in water. Read the upper confidence limit, not the point estimate. The gaps listed at the top (no post-exercise recovery, no face immersion) mean the riskiest situations are not represented.

In [ ]:
def transfer_frame(hr, baseline, subject, context):
    f = hr_features(hr, baseline)
    f['cusum'] = cusum_score(hr)
    f['t'] = np.arange(len(f))
    f['subject'], f['context'], f['onset'], f['target'] = subject, context, -1, 0
    eligible = f[HR1HZ].notna().all(axis=1) & (f.valid30 >= 0.7)
    out = f[eligible].reset_index(drop=True)
    recordings_transfer[subject] = {'hr': np.asarray(hr, float), 'baseline': baseline}
    return out

recordings_transfer = {}
swim = pd.read_csv(paths['swim'], parse_dates=['datetime'])
transfer_parts = []
for sid, person in swim.groupby('subject_id'):
    rest = person.loc[person.trial_label.eq('dry_rest'), 'hr_polar_h10_bpm']
    if rest.notna().sum() < 60:
        continue
    baseline = float(rest.median())
    for (trial, label), trial_rows in person.groupby(['trial_code', 'trial_label']):
        trial_rows = trial_rows.sort_values('datetime')
        seconds = (trial_rows.datetime - trial_rows.datetime.iloc[0]).dt.total_seconds().round().astype(int).to_numpy()
        segment_id = np.r_[0, np.cumsum(np.diff(seconds) != 1)]
        for seg in np.unique(segment_id):
            part = trial_rows[segment_id == seg]
            if len(part) < 200:
                continue
            transfer_parts.append(transfer_frame(part.hr_polar_h10_bpm.to_numpy(float), baseline,
                                                 f'swim/{int(sid)}/{trial}/{seg}', f'swimming dataset: {label}'))

workbook = openpyxl.load_workbook(paths['apnea'], read_only=True, data_only=True)
try:
    rows = workbook['FC'].iter_rows(values_only=True)
    header = next(rows)
    cols = [i for i, h in enumerate(header) if re.fullmatch(r'\d{1,2}:\d{2}:\d{2}', str(h))]
    hr_sessions = {int(r[0]): (r[1], np.array([np.nan if r[i] is None else float(r[i]) for i in cols]))
                   for r in rows if r[1] in {'AG', 'CG'} and r[0] is not None}
    rows = workbook['STA_BVP'].iter_rows(values_only=True)
    header = list(next(rows))
    rest_hr = {int(r[0]): r[header.index('Resting_HR (bpm)')] for r in rows if r[1] in {'AG', 'CG'}}
finally:
    workbook.close()
for pid, (group, hr) in hr_sessions.items():
    baseline = pd.to_numeric(rest_hr.get(pid), errors='coerce')
    if not np.isfinite(baseline) or baseline <= 0:
        continue
    context = 'breath-hold dataset: repeated breath-holds (AG)' if group == 'AG' else 'breath-hold dataset: normal breathing (CG)'
    transfer_parts.append(transfer_frame(np.where(hr > 0, hr, np.nan), float(baseline), f'apnea/{pid}', context))
transfer = pd.concat(transfer_parts, ignore_index=True)
print(f'Transfer data: {transfer.subject.nunique()} segments, {len(transfer) / 3600:.2f} h eligible')

In [ ]:
def sustained_below(frame, bpm, hold):
    '''Seconds the 10-s median HR has stayed below bpm, reset at gaps and between subjects.'''
    below = (frame.hr_med10 < bpm).to_numpy()
    same = prepare(frame)['same']
    run, count = np.zeros(len(frame)), 0
    for i in range(len(frame)):
        count = (count + 1 if same[i] else 1) if below[i] else 0
        run[i] = count
    return run

SWEEP_SCORES = {'rule_drop_from_peak': RULES['rule_drop_from_peak'],
                'rule_drop_from_baseline': RULES['rule_drop_from_baseline'],
                'cusum': RULES['cusum']}
person = transfer.subject.str.split('/').str[1]
sweep_rows = []
def add_rows(rule, threshold, values):
    high_all = np.nan_to_num(values, nan=-np.inf) >= threshold
    for context, block in transfer.groupby('context'):
        take = transfer.context.eq(context).to_numpy()
        o = alert_outcome(block.reset_index(drop=True), values[take], threshold)
        hours = len(block) / 3600
        lo, hi = poisson_rate_ci(o['alert_starts'], hours)
        sweep_rows.append({'rule': rule, 'threshold': threshold, 'context': context, 'people': person[take].nunique(),
                           'hours': hours, 'false_alert_starts': o['alert_starts'],
                           'false_alerts_per_hour': o['alert_starts'] / hours, 'ci95_lo': lo, 'ci95_hi': hi,
                           'people_with_alert': person[take & high_all].nunique()})

for rule, scorer in SWEEP_SCORES.items():
    values = scorer(transfer)
    for threshold in SWEEP_GRID[rule]:
        add_rows(rule, threshold, values)
add_rows(f'vasis_hr_below_{VASIS_HR_BPM:g}_for_{VASIS_HOLD_S}s', VASIS_HOLD_S + 1,   # "> 10 s" = 11th second
         sustained_below(transfer, VASIS_HR_BPM, VASIS_HOLD_S))
label_free = pd.DataFrame(sweep_rows)
show = label_free.assign(rate=lambda d: d.false_alerts_per_hour.round(2).astype(str) + ' [' +
                         d.ci95_lo.round(2).astype(str) + '-' + d.ci95_hi.round(2).astype(str) + ']')
display(show.pivot_table(index=['rule', 'threshold'], columns='context', values='rate', aggfunc='first'))
display(label_free.groupby('context')[['people', 'hours']].max().round(2))

### Learned and threshold-tuned models on the same data

*Runs only with investigator/clinician labels.* The final models are fitted on 75% of the labelled patients, with thresholds chosen on the remaining 25% (stratified). They are then frozen and applied to the false-alarm data above. The HRV models need beat timing, which these 1 Hz sources lack, so they are reported as unavailable.

In [ ]:
if not LABELS_AVAILABLE:
    print('Skipped: no investigator/clinician labels supplied (see the Labels section).')
else:
    final_tune = set()
    all_ids = np.array(sorted(labels.subject))
    for has_event in [True, False]:
        pool = [s for s in all_ids if bool(subjects.loc[s, 'event']) == has_event]
        final_tune |= set(np.random.default_rng(SEED + 1).permutation(pool)[:max(1, round(0.25 * len(pool)))])
    final_fit = anchors[~anchors.subject.isin(final_tune)]
    final_tune_frame = anchors[anchors.subject.isin(final_tune)].reset_index(drop=True)
    final_models, final_thresholds = {}, {}
    for m in MODELS:
        final_models[m] = None if m in RULES else fit_model(m, final_fit)
        final_thresholds[m] = None if m == 'no_alert' else choose_threshold(final_tune_frame, score(m, final_models[m], final_tune_frame))
    print('Final thresholds:', {k: (None if v is None else round(v, 4)) for k, v in final_thresholds.items()})
    transfer_rows = []
    for m in MODELS:
        if m in LEARNED and LEARNED[m] != 'window' and any(c in HRV for c in LEARNED[m]):
            transfer_rows.append({'model': m, 'context': 'all', 'note': 'needs beat timing; not available at 1 Hz'})
            continue
        values = score(m, final_models[m], transfer, recordings_transfer)
        th = final_thresholds[m]
        high_all = np.zeros(len(transfer), bool) if th is None else np.nan_to_num(values, nan=-np.inf) >= th
        person = transfer.subject.str.split('/').str[1]
        for context, block in transfer.groupby('context'):
            take = transfer.context.eq(context).to_numpy()
            o = alert_outcome(block.reset_index(drop=True), values[take], th)
            transfer_rows.append({'model': m, 'context': context, 'people': person[take].nunique(),
                                  'hours': len(block) / 3600, 'false_alert_starts': o['alert_starts'],
                                  'ci95_lo': poisson_rate_ci(o['alert_starts'], len(block) / 3600)[0],
                                  'ci95_hi': poisson_rate_ci(o['alert_starts'], len(block) / 3600)[1],
                                  'false_alerts_per_hour': o['alert_starts'] / (len(block) / 3600),
                                  'warning_fraction': o['warning_fraction'],
                                  'people_with_alert': person[take & high_all].nunique()})
    transfer_results = pd.DataFrame(transfer_rows)
    display(transfer_results.pivot_table(index='model', columns='context', values='false_alerts_per_hour').round(2))

## Figures

In [ ]:
def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / f'{name}.png', dpi=130, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# 1. Review candidates (not labels), for clinician adjudication.
events = (candidates[candidates.review_candidate_onset_s.notna()]
          .rename(columns={'review_candidate_onset_s': 'onset_s', 'review_candidate_min': 'onset_min',
                           'sbp_fall_near_candidate': 'bp_confirmed'}).sort_values('subject'))
ncol = 4
nrow = max(1, math.ceil(len(events) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 2.1 * nrow), squeeze=False)
for ax, row in zip(axes.flat, events.itertuples()):
    rec = recordings[row.subject]
    onset = int(row.onset_s)
    lo, hi = max(0, onset - 600), min(len(rec['hr']), onset + 300)
    x = (np.arange(lo, hi) - onset) / 60
    ax.plot(x, pd.Series(rec['hr']).rolling(10, min_periods=6).median().to_numpy()[lo:hi], color='tab:blue', lw=.8)
    ax2 = ax.twinx()
    ax2.plot(x, rec['sbp_ratio'][lo:hi], color='tab:red', lw=.6)
    ax2.set_ylim(0.3, 1.4)
    ax.axvline(0, color='black', lw=.8)
    ax.set_title(f"{row.subject} @ {row.onset_min:.1f} min{' SBP↓' if row.bp_confirmed else ''}", fontsize=8)
for ax in axes.flat[len(events):]:
    ax.axis('off')
fig.suptitle('Review candidates, NOT labels: HR (blue) and artifact-filtered SBP ratio (red); minutes from candidate', y=1.0)
save(fig, '01_candidate_review')

# 5. Label-free false-alarm rates by threshold (95% CI).
rules = list(SWEEP_SCORES)
fig, axes = plt.subplots(1, len(rules), figsize=(5 * len(rules), 4.2), sharey=True)
for ax, rule in zip(axes, rules):
    block = label_free[label_free.rule.eq(rule)]
    for context, rows in block.groupby('context'):
        err = np.vstack([rows.false_alerts_per_hour - rows.ci95_lo, rows.ci95_hi - rows.false_alerts_per_hour])
        ax.errorbar(rows.threshold, rows.false_alerts_per_hour, yerr=err, marker='o', ms=3, capsize=2, label=context)
    ax.set(title=rule, xlabel='threshold', yscale='symlog')
axes[0].set_ylabel('False alert starts / hour (no loss of consciousness)')
axes[-1].legend(fontsize=6)
save(fig, '05_label_free_false_alarms')

In [ ]:
if not LABELS_AVAILABLE:
    print('Skipped: no investigator/clinician labels supplied (see the Labels section).')
else:
    # 2. Event recall vs unmatched alert burden (cross-validated, locked thresholds).
    fig, ax = plt.subplots(figsize=(8, 5))
    block = results[results.subset.eq('all events')]
    for r in block.itertuples():
        ax.scatter(r.unmatched_per_hour, r.event_recall, s=40)
        ax.annotate(r.model, (r.unmatched_per_hour, r.event_recall), fontsize=7)
    ax.axvline(MAX_UNMATCHED_PER_HOUR, ls=':', color='gray')
    ax.set(xlabel='Unmatched alert starts / hour (held-out folds)', ylabel='Events credited / events',
           title='Cross-validated alert performance (validation-chosen thresholds)', ylim=(-.03, 1.03))
    save(fig, '02_recall_vs_burden')

    # 3. Lead times of credited alerts.
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, m in enumerate(MODELS):
        leads = []
        for fold in range(N_FOLDS):
            take = anchors.fold.eq(fold).to_numpy()
            leads += alert_outcome(anchors[take].reset_index(drop=True), oof[m][take], lookup.get((fold, m)))['leads']
        ax.scatter(leads, np.full(len(leads), i), s=14)
    ax.axvline(0, color='black', lw=.8)
    ax.set_yticks(range(len(MODELS)), MODELS)
    ax.set(xlabel='Seconds of warning before onset (negative = late)', title='Lead time of credited alerts')
    save(fig, '03_lead_times')

    # 4. False alarms when transferred.
    pivot = transfer_results.dropna(subset=['false_alerts_per_hour']).pivot_table(index='model', columns='context', values='false_alerts_per_hour')
    fig, ax = plt.subplots(figsize=(11, 5))
    pivot.plot.barh(ax=ax)
    ax.set(xlabel='False alert starts / hour (no loss of consciousness)', title='Frozen tilt models on swimming and breath-hold HR')
    ax.legend(fontsize=7)
    save(fig, '04_transfer_false_alarms')

## Export

In [ ]:
configuration = {
    'purpose': 'heart-rate track: label-free false-alarm test; classifier only with investigator/clinician labels',
    'version': 'v3 (2026-11-09)', 'deployable': False, 'blackout_probability': False,
    'labels': {'accepted': 'ADJUDICATED_EVENTS_CSV rows with a stated source (investigator outcome or clinician)',
               'self_derived_labels_used': False, 'labelled_recordings': int(len(labels)),
               'labelled_events': int(labels.event.sum()) if len(labels) else 0,
               'classifier_trained': bool(LABELS_AVAILABLE),
               'review_candidate_rule': {'fall_ratio': HR_COLLAPSE_FALL, 'hold_s': HR_COLLAPSE_HOLD_S,
                                         'search_s': SEARCH, 'note': 'for clinician review only, not a label'}},
    'label_free_test': {'vasis_reference': {'hr_bpm': VASIS_HR_BPM, 'hold_s': VASIS_HOLD_S}, 'sweep_grid': SWEEP_GRID},
    'horizon_s': HORIZON_S, 'late_grace_s': LATE_GRACE_S, 'credit_lead_s': CREDIT_LEAD_S,
    'budget': {'unmatched_per_hour': MAX_UNMATCHED_PER_HOUR, 'warning_fraction': MAX_WARNING_FRACTION},
    'features': {'hr1hz': HR1HZ, 'hrv': HRV, 'window_bins': WINDOW_BINS}, 'input_rate_hz': 1, 'seed': SEED,
    'limitations': ['no public syncope outcomes or event times for the tilt data',
                    'false-alarm data lack post-exercise recovery and face-immersion apnoea',
                    'clinical syncope patients on land, not swimmers', 'ECG-derived HR, not chest MAX30102 optical HR',
                    'tilt/GTN timing assumed from protocol, not annotated', 'no hypoxic blackout examples'],
    'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                 'scikit_learn': sklearn.__version__},
}
candidates.to_csv(OUT / 'review_candidates.csv', index=False)
recording_audit.to_csv(OUT / 'recording_audit.csv', index=False)
label_free.to_csv(OUT / 'label_free_false_alarms.csv', index=False)
if LABELS_AVAILABLE:
    configuration['final_thresholds'] = final_thresholds
    joblib.dump({'models': {k: v for k, v in final_models.items() if v is not None}, 'thresholds': final_thresholds,
                 'features': {'hr1hz': HR1HZ, 'hrv': HRV}}, OUT / 'hr_syncope_models.joblib')
    labels.to_csv(OUT / 'labels_used.csv', index=False)
    results.to_csv(OUT / 'cv_results.csv', index=False)
    thresholds.to_csv(OUT / 'fold_thresholds.csv', index=False)
    transfer_results.to_csv(OUT / 'transfer_false_alarms.csv', index=False)
    pd.DataFrame({'subject': anchors.subject, 't': anchors.t, 'target': anchors.target, 'fold': anchors.fold,
                  **{f'score_{m}': oof[m] for m in MODELS}}).to_csv(OUT / 'oof_predictions.csv.gz', index=False)
    reloaded = joblib.load(OUT / 'hr_syncope_models.joblib')
    probe = final_tune_frame.head(5)
    assert np.allclose(reloaded['models']['logistic'].predict_proba(probe[HR1HZ].to_numpy(float)),
                       final_models['logistic'].predict_proba(probe[HR1HZ].to_numpy(float)))
(OUT / 'hr_syncope_configuration.json').write_text(json.dumps(configuration, indent=2, default=str), encoding='utf-8')
print('Exported:', sorted(p.name for p in OUT.iterdir() if p.is_file()))

## Reading the results

- **Without labels, read only the label-free table.** It shows how often each rule would false-alarm during swimming and breath-holds at each threshold. A threshold whose *upper* 95% limit is above your alarm budget is not shown to be safe. The VASIS row shows what a published cardioinhibitory criterion does there.
- **Review candidates are not results.** They are a worklist for a clinician. Use them to build `ADJUDICATED_EVENTS_CSV`, together with the investigators' outcome table if they share it.
- **With labels:**
  - Compare the learned models with the simple rules at the same alert burden.
  - Read lead time: credits with lead ≤ 0 are detections, not forecasts.
  - Each recall and rate comes with a 95% interval. Overlapping intervals mean no demonstrated difference.
- **HRV versus 1 Hz.** If `*_hrv` clearly beats the 1 Hz models on labelled data, that justifies on-device beat detection from the 100 Hz PPG.
- **Before any claim:** validate chest MAX30102 HR against ECG in water (notebook 02), collect HR through post-exercise recovery and face-immersion breath-holds, and evaluate prospectively.